# UdaPlay 01 Solution Project

## Purpose

This notebook addresses the reviewer feedback by:

1. Using the course-provided `lib.agents.Agent` wrapper.
2. Registering the required tools:
   - `retrieve_game`
   - `evaluate_retrieval`
   - `game_web_search`
3. Local video game dataset for vector database retrieval.
4. Stateful multi-turn demo using the same `session_id`.
5. Conversational memory with pronoun resolution using **"it"**.

### 1. Imports

In [25]:
from dotenv import load_dotenv
import os

load_dotenv()

from lib.udaplay_vector_store import UdaPlayVectorStore
from lib.udaplay_tools import ( 
    configure_udaplay_tools,
    retrieve_game,
    evaluate_retrieval,
    game_web_search,
)
from lib.agents import Agent
import lib.llm

In [26]:
pip install sentence_transformers


[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


### 2. Vector Store

In [27]:

vector_store = UdaPlayVectorStore(
    persist_dir="./chroma_db/udaplay_games",
    collection_name="udaplay_games",
    reset_collection=False,
)

print("Vector store initialized.")


Vector store initialized.


### 3. Loading video game dataset

In [28]:
vector_store.add_games_from_file("data/games.json")
print("Local game dataset loaded into store.")

Local game dataset loaded into store.


### 4. Configure UdaPlay tools

In [29]:
configure_udaplay_tools(vector_store)
print("UdaPlay tools configured.")

UdaPlay tools configured.


### 5 Agent wrapper

In [30]:
agent = Agent(
    model_name="gpt-4o-mini",
    tools=[
        retrieve_game,
        evaluate_retrieval,
        game_web_search,
    ],
    instructions=
    """
You are UdaPlay, a helpful video game assistant. You have access to the following tools:

1. retrieve_game
   - Use this first when the user asks about video game facts that may exist in the local dataset.
2. evaluate_retrieval
   - Use this after retrieval to determine whether the retrieved context is sufficient to answer the user.
3. game_web_search
   - Use this only when local retrieval is insufficient, incomplete, or when the user asks about latest, current, recent, or ongoing information.
Also, maintain conversational memory across the same session_id, resolve pronouns and references such as "it", "that game", "this title", and "the game", prefer internal over web search,  use web search as fall back, when web search is used, include URLs where available.
"""
)

print("UdaPlay Agent created using lib.agents.Agent.")


UdaPlay Agent created using lib.agents.Agent.


### 6 Function to  agent responses

In [31]:
def print_agent_response(label, response):
    print("=" * 100)
    print(label)
    print("=" * 100)

    final_state = response.get_final_state()

    messages = final_state.get("messages", [])
    total_tokens = final_state.get("total_tokens", 0)

    print("\nAnswer:")
    if messages:
        print(messages[-1].content)
    else:
        print("No messages found.")

    print("\nTool Usage Trace:")
    tools_used = []

    for m in messages:
        tool_calls = getattr(m, "tool_calls", None)

        if tool_calls:
            for call in tool_calls:
                tool_name = call.function.name
                tools_used.append(tool_name)
                print(f"- AI requested tool: {tool_name}")
                print(f"  Arguments: {call.function.arguments}")

        if getattr(m, "role", None) == "tool":
            tool_name = getattr(m, "name", "unknown_tool")
            print(f"- Tool returned result from: {tool_name}")
            print(f"  Result preview: {m.content[:500]}")

    print("\nTools Used:")
    print(tools_used if tools_used else "No tools used in this run.")

    print("\nTotal Tokens:")
    print(total_tokens)

    print("\n")


### 7. Stateful memory across the session and using pronouns


In [32]:
session_id = "udaplay_memory_demo_01"

response1 = agent.invoke(
    "Who developed FIFA 21?",
    session_id=session_id
)
print("Session Id: " + session_id)
print_agent_response("Query 1: Who developed FIFA 21?", response1)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session Id: udaplay_memory_demo_01
Query 1: Who developed FIFA 21?

Answer:
FIFA 21 was developed by EA Sports, specifically by EA Vancouver and EA Romania. The game was released on October 9, 2020. You can find more information about it on [Wikipedia](https://en.wikipedia.org/wiki/FIFA_21).

Tool Usage Trace:
- AI requested tool: retrieve_game
  Arguments: {"query":"FIFA 21 developer"}
- Tool returned result from: retrieve_game
  Result preview: {"tool": "retrieve_game", "query": "FIFA 21 developer", "top_k": 5, "results": [{"id": "game_97c9a9a6e97c3d63",

In [34]:
response2 = agent.invoke(
    "What platform was it released on?",
    session_id=session_id
)
print("Session Id: " + session_id)
print_agent_response("Query 2: What platform was it released on?", response2)


[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
Session Id: udaplay_memory_demo_01
Query 2: What platform was it released on?

Answer:
FIFA 21 was released on the following platforms:

- Microsoft Windows
- PlayStation 4
- Xbox One
- Nintendo Switch

Additionally, enhanced versions for PlayStation 5 and Xbox Series X/S were released on December 3, 2020.

Tool Usage Trace:
- AI requested tool: retrieve_game
  Arguments: {"query":"FIFA 21 developer"}
- Tool returned result from: retrieve_game
  Result preview: {"tool": "retrieve_game", "query": "FIFA 21 developer", "top_k": 5, "results": [{"id": "game_97c9a9a6e97c3d63", "document": "Title: FIFA 21\nDeveloper: EA Vancouver and EA Romania\nPublisher: Electronic Arts\nRelease Date: October 9, 2020\nPlatforms: Microsoft Windows, PlayStation 4, Xbox One, Nintendo Switch\nGenre: Sports\nDescription: FIFA 21 is a football simu

### Rubric Tracebility

- `lib.agents.Agent` wrapper is used.
- internal retrieval, retrieval evaluation, and web search fallback tools.
- `session_id` is used across 
- Pronoun :  **"it"**
